In [2]:
!python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/          
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/
!python -m pip install "paddleocr[all]"                           
!pip install torch transformers                                      
!python -m pip install label-studio 
!pip install transformers datasets pillow torchvision
# ref website https://www.paddlepaddle.org.cn/en/install/quick?docurl=/documentation/docs/en/develop/install/pip/windows-pip_en.html

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cpu/
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.2
    Uninstalling protobuf-3.20.2:
      Successfully uninstalled protobuf-3.20.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlepaddle-gpu 2.6.2 requires protobuf<=3.20.2,>=3.1.0; platform_system == "Windows", but you have protobuf 6.33.6 which is incompatible.


In [ ]:
import json
from paddleocr import PaddleOCR
import paddle
import os
import pprint
import torch
from transformers import LayoutLMv3Tokenizer, LayoutLMv3ForTokenClassification
from PIL import Image
from torchvision import transforms

In [ ]:
!setx CUDA_HOME "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.8"
!setx PATH "C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v11.8\bin;%PATH%"

# Run the code in gpu

### To check does the gpu is decteced or not 

In [ ]:
print(paddle.device.get_device())

# Extracting the text and bbox from the image using paddleocr


In [ ]:
image_path = "./training data/1.jpg"
paddle_ocr_result_path = "output/paddle_ocr_result.json"

####              GPU

In [1]:
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

ocr = PaddleOCR(use_angle_cls=True, use_gpu=True)
result = ocr.predict(image_path)

NameError: name 'PaddleOCR' is not defined

#### Processor

In [ ]:
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

ocr = PaddleOCR(
    use_doc_orientation_classify=False, 
    use_doc_unwarping=False, 
    use_textline_orientation=False) # text detection + text recognition


# image_path = "./training data/1.jpg"

result = ocr.predict(image_path) # predict the image 

#### Dump to json file

In [ ]:
for res in result:
    res.print() # print the result of each image
    # res.save_to_img("output")
    # res.save_to_json("output/paddle_ocr_result.json")
    res.save_to_json(paddle_ocr_result_path)

# Convert ocr formate to label studio formate

In [ ]:
def convert_ocr_to_ls(ocr_json, image_url, img_w, img_h):

    results = []

    polys = ocr_json["rec_polys"]
    texts = ocr_json["rec_texts"]

    for i, poly in enumerate(polys):

        xs = [p[0] for p in poly]
        ys = [p[1] for p in poly]

        x_min = min(xs)
        y_min = min(ys)

        x_max = max(xs)
        y_max = max(ys)

        w = x_max - x_min
        h = y_max - y_min

        results.append({
            "id": f"box_{i}",
            "type": "rectangle",
            "from_name": "bbox",
            "to_name": "image",
            "original_width": img_w,
            "original_height": img_h,
            "image_rotation": 0,
            "value": {
                "x": x_min/img_w*100,
                "y": y_min/img_h*100,
                "width": w/img_w*100,
                "height": h/img_h*100,
                "rotation": 0
            }
        })
        
        # OCR Text
        results.append({
            "id": f"box_{i}",
            "type": "textarea",
            "from_name": "transcription",
            "to_name": "image",
            "value": {
                "text": [texts[i]]
            }
        })
        
    return {
        "data": {
            "ocr": image_url
        },
        "predictions":[
            {
                "model_version":"paddleocr",
                "result": results
            }
        ]
    }


In [ ]:
img_path_ls="http://localhost:8000/output/1.jpg"
ocr_to_ls_path="output/labelstudio_tasks.json"

In [ ]:
data=convert_ocr_to_ls(res, img_path_ls, 2232, 3300)
pprint.pprint(data)
                       
with open(ocr_to_ls_path, "w") as f:
    json.dump(data, f, indent=4)

    
    # with open("./output/paddle_ocr_result.json", "r") as f:
#     data = json.load(f)
#     print(data.get('ocr'))

# Load the labeled data of label studio

In [ ]:
label_studi_result_path="output/labelstudio_output.json"

In [ ]:
with open(label_studi_result_path, 'r') as f:
    data = json.load(f)
pprint.pprint(data)

# Convert label studio formate to layoutlvm3 formate [BIO Formate]